In [1]:
import numpy as np

def all_close(a_, b_, atol=1e-10, rtol=1e-10):
    # --- compare per level ---
    for k in range(depth + 1):

        x = a_[k]
        y = b_[k]
        if x is None:
            x = np.zeros_like(y)
        delta = x - y
        print(f"level {k} abs: ", abs(delta).max() < atol) if hasattr(delta, "max") else abs(delta) < atol
        rel_dist = (abs(delta) / abs(x)).max() if hasattr(delta, "max") else abs(delta) / abs(x)
        print(f"level {k} rel: ", rel_dist < rtol if rel_dist < rtol else rel_dist)


B, L, d, depth = 1000, 101, 3, 7

np.random.seed(12335)
path_a = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_a = np.concat([np.zeros((B, 1, d)), np.cumsum(path_a, axis=1)], axis=1)

np.random.seed(23466)
path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

In [2]:
from tensordev import Universal

import numpy as np

NumpyCore = Universal(np)

In [3]:
sig_a = NumpyCore.tensor_path_signature(path_a, trunc=depth)
len(sig_a), [lvl.shape for lvl in sig_a]

(8,
 [(1000, 1),
  (1000, 3),
  (1000, 9),
  (1000, 27),
  (1000, 81),
  (1000, 243),
  (1000, 729),
  (1000, 2187)])

In [4]:
sig_a = NumpyCore.tensor_path_signature(path_a, trunc=depth)
len(sig_a), [lvl.shape for lvl in sig_a]

path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

sig_b = NumpyCore.tensor_development([path_b], trunc=depth)
sig_c = NumpyCore.tensor_development([path_c], trunc=depth, block_size=100)

sig_c_ = NumpyCore.tensor_product(sig_a, sig_b, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_a[i], sig_c[i][:, 0]) # check if first block gives sig_a
    assert np.allclose(sig_c_[i], sig_c[i][:, 1]) # check if second block gives sig_a \otimes sig_b

In [5]:
from tensordev import Jax
JaxCore = Jax()

In [6]:
sig_a = JaxCore.tensor_path_signature(path_a, trunc=depth)
sig_b = JaxCore.tensor_path_signature(path_b, trunc=depth)
sig_c = JaxCore.tensor_path_signature(path_c, trunc=depth, block_size=100)

In [10]:
sig_c_ = JaxCore.tensor_product(sig_a, sig_b, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_a[i], sig_c[i][:, 0]) # check if first block gives sig_a
    assert np.allclose(sig_c_[i], sig_c[i][:, 1]) # check if second block gives sig_a \otimes sig_b

In [8]:
import signax  
sig_a_sx = signax.signature(path_a, depth=depth)
sig_b_sx = signax.signature(path_b, depth=depth)
sig_c_sx = signax.signature(path_c, depth=depth)

In [9]:
sig_a_sx_ = JaxCore.tensor_from_flat(sig_a_sx, dim=d, insert_zero_level=1.0)
sig_b_sx_ = JaxCore.tensor_from_flat(sig_b_sx, dim=d, insert_zero_level=1.0)
sig_c_sx_ = JaxCore.tensor_from_flat(sig_c_sx, dim=d, insert_zero_level=1.0)

sig_c_sx_prod = JaxCore.tensor_product(sig_a_sx_, sig_b_sx_, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_c_sx_prod[i], sig_c_sx_[i]) # check if second block gives sig_a \otimes sig_b